# E6 | Model Anomaly Detection — Isolation Forest
Detectar incidentes anômalos fora dos padrões normais (complementa K-Means)

## 📋 Objetivo

Detectar incidentes **anômalos** que desviam significativamente do padrão normal, usando **Isolation Forest**.

### Por que Isolation Forest?
- **K-Means** agrupa incidentes normais em clusters A/B/C/D
- **Isolation Forest** detecta outliers (anomalias) independente de clusters
- Complementar: Incidente pode estar no cluster D E ainda ser anômalo
- Útil para alertas preventivos: "este incidente é 3σ fora do normal"

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import mlflow
import joblib

load_dotenv()
print('✅ Setup OK')

✅ Setup OK


In [2]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

query = '''
    SELECT 
        duracao_horas,
        pct_violacao_sla,
        count_reaberturas,
        count_comentarios,
        custo_estimado,
        EXTRACT(DOW FROM data_abertura)::INT AS dia_semana,
        EXTRACT(HOUR FROM data_abertura)::INT AS hora_abertura
    FROM gold_ml.ml_clustering_dataset
    WHERE duracao_horas IS NOT NULL
    AND pct_violacao_sla IS NOT NULL
'''
df = pd.read_sql(query, engine)

print(f'✅ Carregados {len(df)} incidentes para detecção de anomalias')
print(f'   Features: {df.columns.tolist()}')

ProgrammingError: (psycopg2.errors.UndefinedTable) relation "gold_ml.ml_clustering_dataset" does not exist
LINE 10:     FROM gold_ml.ml_clustering_dataset
                  ^

[SQL: 
    SELECT 
        duracao_horas,
        pct_violacao_sla,
        count_reaberturas,
        count_comentarios,
        custo_estimado,
        EXTRACT(DOW FROM data_abertura)::INT AS dia_semana,
        EXTRACT(HOUR FROM data_abertura)::INT AS hora_abertura
    FROM gold_ml.ml_clustering_dataset
    WHERE duracao_horas IS NOT NULL
    AND pct_violacao_sla IS NOT NULL
]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [ ]:
mlflow.set_experiment('isolation_forest_anomaly_detection')

with mlflow.start_run(run_name='isolation_forest_v1'):
    X = df.copy()
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    contamination = 0.05
    model = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=100,
        max_samples='auto',
        n_jobs=-1
    )
    
    anomaly_labels = model.fit_predict(X_scaled)
    anomaly_scores = model.score_samples(X_scaled)
    
    df['anomaly'] = anomaly_labels
    df['anomaly_score'] = anomaly_scores
    
    n_anomalies = (anomaly_labels == -1).sum()
    pct_anomalies = 100 * n_anomalies / len(df)
    
    mlflow.log_params({
        'contamination': contamination,
        'n_estimators': 100,
        'max_samples': 'auto',
        'n_features': X.shape[1]
    })
    
    mlflow.log_metrics({
        'n_anomalies_detected': float(n_anomalies),
        'pct_anomalies': pct_anomalies,
        'mean_anomaly_score': float(anomaly_scores.mean()),
        'std_anomaly_score': float(anomaly_scores.std())
    })
    
    mlflow.set_tag('model_type', 'IsolationForest')
    mlflow.set_tag('task', 'anomaly_detection')
    mlflow.set_tag('use_case', 'detect_abnormal_incidents')
    
    joblib.dump(scaler, 'scaler_if.pkl')
    mlflow.log_artifact('scaler_if.pkl', 'preprocessing')
    
    mlflow.sklearn.log_model(model, 'isolation_forest_model')
    
    print(f'✅ Modelo treinado')
    print(f'   Anomalias detectadas: {n_anomalies} ({pct_anomalies:.1f}%)')
    print(f'   Score médio: {anomaly_scores.mean():.4f} ± {anomaly_scores.std():.4f}')

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if label == -1 else 'blue' for label in anomaly_labels]
ax.scatter(X_2d[:, 0], X_2d[:, 1], c=colors, alpha=0.5, s=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('Anomalias detectadas por Isolation Forest (PCA 2D)')
ax.legend(['Normal', 'Anomalia'], loc='best')
fig.tight_layout()
fig.savefig('anomaly_detection_pca.png', dpi=100, bbox_inches='tight')
mlflow.log_artifact('anomaly_detection_pca.png', 'evaluation')
plt.close()

print('✅ PCA visualization salva')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(anomaly_scores[anomaly_labels == 1], bins=50, alpha=0.7, label='Normal', color='blue')
ax.hist(anomaly_scores[anomaly_labels == -1], bins=20, alpha=0.7, label='Anomalia', color='red')
ax.set_xlabel('Anomaly Score (mais negativo = mais anômalo)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de Anomaly Scores')
ax.legend()
ax.axvline(model.offset_, color='orange', linestyle='--', linewidth=2, label='Threshold')
fig.tight_layout()
fig.savefig('anomaly_scores_distribution.png', dpi=100, bbox_inches='tight')
mlflow.log_artifact('anomaly_scores_distribution.png', 'evaluation')
plt.close()

print('✅ Distribuição de scores salva')

In [ ]:
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\anomaly_detection')
base_path.mkdir(parents=True, exist_ok=True)

df_anomalies = df[df['anomaly'] == -1].copy()
df_anomalies.to_csv(str(base_path / 'detected_anomalies.csv'), index=False)

anomaly_summary = pd.DataFrame({
    'metric': ['total_incidents', 'anomalies_detected', 'pct_anomalies', 'mean_anomaly_score', 'min_anomaly_score', 'max_anomaly_score'],
    'value': [
        len(df),
        n_anomalies,
        pct_anomalies,
        anomaly_scores.mean(),
        anomaly_scores.min(),
        anomaly_scores.max()
    ]
})
anomaly_summary.to_csv(str(base_path / 'anomaly_summary.csv'), index=False)

print(f'✅ Salvos em {base_path}')
print(f'\n📊 Resumo:')
print(anomaly_summary.to_string(index=False))